In [31]:
import openpyxl as wb
import pandas as pd
from geopy.geocoders import Nominatim
import folium
from geopy.extra.rate_limiter import RateLimiter

In [32]:
planilhaLinks = wb.load_workbook("SPTC_12Meses_2026-03-06.XLSX")
#print(planilhaLinks.sheetnames)
aba = planilhaLinks["PA_em_05-03-2026"]
colunas = [3,5,6,29]
dados = []
for row in aba.iter_rows(values_only=True):
    linhas = [row[i-1] for i in colunas if i-1 <len(row)]
    if any(linhas):
        dados.append(linhas)
df = pd.DataFrame(dados[1:], columns=dados[0])
df.columns = ["Unidade", "CEP","Endereço", "IP"]
print(df.head(10))
   
    

                                             Unidade  CEP  \
0        EQUIPE DE PERICIA CRIMINALISTICA-ADAMANTINA  676   
1         EQUIPE DE PERICIA CRIMINALISTICA-AMERICANA  100   
2         EQUIPE DE PERICIA CRIMINALISTICA-ANDRADINA  871   
3             EQUIPE DE PERICIA CRIMINALISTICA-ASSIS  609   
4             EQUIPE DE PERICIA CRIMINALISTICA-AVARE   11   
5          EQUIPE DE PERICIA CRIMINALISTICA-BARRETOS  808   
6          EQUIPE DE PERICIA CRIMINALISTICA-BARRETOS  630   
7         EQUIPE DE PERICIA CRIMINALISTICA-BEBEDOURO  898   
8          EQUIPE DE PERICIA CRIMINALISTICA-BOTUCATU  374   
9  EQUIPE DE PERICIA CRIMINALISTICA-BRAGANCA PAUL...    2   

                                            Endereço      IP  
0                                                     432.65  
1                                                     605.71  
2                                                     432.65  
3                                                     432.65  
4            

In [33]:
geolocator = Nominatim(user_agent="meu_app")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
df['endereco_completo'] = df["Endereço"] + ', ' + df['CEP'] + ', São Paulo, Brasil'
df['local'] = df['endereco_completo'].apply(geocode)
df['latitude'] = df['local'].apply(lambda x: x.latitude if x else None)
df['longitude'] = df['local'].apply(lambda x: x.longitude if x else None)


RateLimiter caught an error, retrying (0/2 tries). Called with (*(', 898, São Paulo, Brasil',), **{}).
Traceback (most recent call last):
  File "c:\Users\faustto.fro\Aulas\hashtg\Lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
  File "c:\Users\faustto.fro\Aulas\hashtg\Lib\site-packages\urllib3\connection.py", line 571, in getresponse
    httplib_response = super().getresponse()
  File "C:\Users\faustto.fro\.pyenv\pyenv-win\versions\3.13.1\Lib\http\client.py", line 1428, in getresponse
    response.begin()
    ~~~~~~~~~~~~~~^^
  File "C:\Users\faustto.fro\.pyenv\pyenv-win\versions\3.13.1\Lib\http\client.py", line 331, in begin
    version, status, reason = self._read_status()
                              ~~~~~~~~~~~~~~~~~^^
  File "C:\Users\faustto.fro\.pyenv\pyenv-win\versions\3.13.1\Lib\http\client.py", line 292, in _read_status
    line = str(self.fp.readline(_MAXLINE + 1), "iso-8859-1")
               ~~~~~~~~~~~~~~~~^^^^^

In [34]:
m = folium.Map(location=[-23.5505, -46.6333], zoom_start=11)
for idx, row in df.dropna(subset=['latitude']).iterrows():
    folium.Marker(
        [row['latitude'], row['longitude']],
        popup=f"{row['Endereço']}<br>CEP: {row['CEP']}",
        tooltip=row['Unidade'] + row['IP']
    ).add_to(m)

m.save('mapa_enderecos.html') 
m


TypeError: can only concatenate str (not "float") to str